In [15]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
import sys
import matplotlib.pyplot as plt

os.chdir("/home/patrick/ansermodelling")

from models.FFNN_network import FFNN
from data.anser_dataset import *
from models.train import *
from data.reparametrisation import normal_to_angles_t
from models.model_wrappers import NNSolver
from models.eval import report_error_stats

In [2]:
train_loader, test_loader = make_dataloaders("data/dataset.npz", normalise_data = True)

In [3]:
#NEEDED TO CONVERT BACK TO 5D 
class AngleLoader:
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for x, y6 in self.loader:
            yield x, normal_to_angles_t(y6)
    def __len__(self):
        return len(self.loader)

In [4]:
train_loader = AngleLoader(train_loader)
test_loader  = AngleLoader(test_loader)

In [5]:
model = FFNN(input_dim=8, output_dim=5, hidden_dims=[256,256,256])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

In [6]:
history = train(model, train_loader, test_loader, optimizer, epochs=200, loss_fn = loss_fn, print_losses = True)

epoch 	 LR 	 Training Loss 	 Test Loss 	 e_p test 	 e_n test
1 	 1.00e-03 	 0.5467 	  0.4557 	 83.5459 	 51.6866
2 	 1.00e-03 	 0.4081 	  0.3711 	 74.6544 	 41.5183
3 	 1.00e-03 	 0.3406 	  0.3086 	 66.2194 	 36.1243
4 	 1.00e-03 	 0.2946 	  0.2732 	 63.5962 	 33.5055
5 	 1.00e-03 	 0.2661 	  0.2542 	 61.0723 	 31.4716
6 	 1.00e-03 	 0.2485 	  0.2533 	 56.2119 	 29.8293
7 	 1.00e-03 	 0.2377 	  0.2268 	 57.8898 	 29.8643
8 	 1.00e-03 	 0.2254 	  0.2132 	 60.7448 	 27.5206
9 	 1.00e-03 	 0.2179 	  0.2158 	 61.8675 	 28.1079
10 	 1.00e-03 	 0.2102 	  0.2156 	 57.6058 	 26.8541
11 	 1.00e-03 	 0.2023 	  0.2002 	 52.4652 	 25.8015
12 	 1.00e-03 	 0.1977 	  0.2028 	 53.1736 	 26.4415
13 	 1.00e-03 	 0.1929 	  0.2029 	 52.0591 	 25.1164
14 	 1.00e-03 	 0.1874 	  0.1998 	 57.7058 	 25.5601
15 	 1.00e-03 	 0.1832 	  0.1904 	 57.5851 	 24.7093
16 	 1.00e-03 	 0.1796 	  0.1917 	 50.2539 	 24.3278
17 	 1.00e-03 	 0.1761 	  0.1772 	 53.0405 	 23.6206
18 	 1.00e-03 	 0.1735 	  0.1936 	 54.7931 	 24

In [8]:
torch.save(model.state_dict(), "models/checkpoints/nn_base.pt")

In [9]:
test_set = np.load("data/test_set.npz")
measurements = test_set["xs"]
poses = test_set["ys"]

In [12]:
nn = NNSolver(model)

In [13]:
%%time
poses_pred_nn, success_nn = nn.solve(measurements)

CPU times: user 419 ms, sys: 23.8 ms, total: 443 ms
Wall time: 88.1 ms


In [16]:
print("Base Model (5d, mse loss, no scheduler)")
report_error_stats(poses_pred_nn,poses,success_nn)

Base Model (5d, mse loss, no scheduler)
Mean pos error: 135, mean angle error: 87.1
Median pos error: 129, Median angle error: 86.3
95% pos error : 247 95% angle error: 154
LM success rate: 1
Convergence rate: 0
Mean pos error of converged: nan, mean angle error of converged: nan 


/home/patrick/ansermodelling/models/eval.py:30: RuntimeWarning: Mean of empty slice
  print(f"Mean pos error of converged: {ex_conv.mean():.3g}, mean angle error of converged: {en_conv.mean():.3g} ")
/home/patrick/ansermodelling/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
